In [35]:
"""Retrieve Content Shared By Influencers During Baseline

KE & SA
-- start date: '2022-08-21T00:00:00Z'
-- end date: '2023-02-21T00:00:00Z'
"""

import pandas as pd
import os
import glob
import yaml
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, message='.*append.*')


#from tweetple import TweetPle
import sys
sys.path.insert(0, '../../src/utils')
from funcs import *
from tqdm import tqdm
sys.path.insert(0, '../../src/utils')
from general import *

def get_followers_quality(sample, followers, likes, replies, path_save, batch):
    sample['author_id'] = sample['author_id'].astype(str)
    participants = list(sample['author_id'])
    followers['influencer_id'] = followers['influencer_id'].astype(str)

    quality_likes = pd.DataFrame()
    for participant in participants:
        followersp = followers[followers['influencer_id'] == participant]
        likesp = likes[likes['influencer_id'] == participant]
        list_to_look = list(followersp['follower_id'].unique())
        quality_likes = quality_likes.append(
            likesp[likesp['liking_user_id'].isin(list_to_look)]
        )
    quality_likes = quality_likes.rename(
        {'liking_user_username': 'follower_username'},
        axis=1
    )
    quality_likes = quality_likes[[
        'follower_username', 'liking_user_id', 'influencer_id']]
    quality_likes = quality_likes.drop_duplicates()
    quality_likes = quality_likes.rename(
        {'liking_user_id': 'follower_id'}, axis=1)
    quality_likes = quality_likes.reset_index(drop=True)
    quality_likes['liked'] = 1

    quality_replies = pd.DataFrame()
    for participant in participants:
        followersp = followers[followers['influencer_id'] == participant]
        repliesp = replies[replies['influencer_id'] == participant]
        list_to_look = list(followersp['follower_id'].unique())
        quality_replies = quality_replies.append(
            repliesp[repliesp['replier_user_id'].isin(list_to_look)]
        )
    quality_replies = quality_replies[[
        'replier_user_id', 'influencer_id']]
    quality_replies = quality_replies.drop_duplicates()
    quality_replies = quality_replies.rename(
        {'replier_user_id': 'follower_id'}, axis=1)
    quality_replies = quality_replies.reset_index(drop=True)
    quality_replies['replied'] = 1

    followers.rename({'username': 'follower_username'}, axis=1, inplace=True)
    cols_keep = ['follower_username', 'follower_id', 'influencer_id']
    followers = followers[cols_keep]
    followers['influencer_id'] = followers['influencer_id'].astype(str)
    followers = followers.merge(quality_likes, on=cols_keep, how='left')
    followers = followers.merge(
        quality_replies,
        on=['follower_id', 'influencer_id'],
        how='left'
    )
    followers[['replied', 'liked']] = followers[['replied', 'liked']].fillna(0)
    followers['strong'] = np.where(
        (followers.liked == 1) & (followers.replied == 1), 1, 0)
    followers['weak'] = np.where(
        (followers.liked == 1) | (followers.replied == 1), 1, 0)
    followers['weak'] = np.where(followers['strong']== 1, 0, followers['weak'])
    #followers = assign_followers_treatment(sample, followers)
    followers.to_parquet(f'{path_save}followers_ties_rep{batch}.parquet')

    return followers



In [36]:
country = 'KE'
path_save = f'../../data/randomization/{country}/influencers/'

# Read the data:
# Tweets:
batch = ''
tweets = pd.read_parquet(f"{path_save}tweets{batch}.parquet")

# Followers 
followers = pd.read_parquet(f"{path_save}followers{batch}.parquet")

# Likes
likes = pd.read_parquet(f"{path_save}likes{batch}.parquet")

# Replies
replies = pd.read_parquet(f"{path_save}replies{batch}.parquet")

# Participants TW
participants_tw = pd.read_parquet(f'{path_save}/confirmed_influencers{batch}.parquet')
participants_tw['author_id'] = participants_tw['id'].astype(str)
participants_tw = participants_tw.drop(['id'], axis = 1)
usernames_tw = list(participants_tw['handle'])
ids_tw = list(participants_tw['author_id'].astype(str))

get_followers_quality(
        participants_tw,
        followers,
        likes,
        replies,
        path_save, 
        batch = batch
    )

# Replicates: YES

df = pd.read_parquet(f'{path_save}followers_ties{batch}.parquet')
df1 = pd.read_parquet(f'{path_save}followers_ties_rep{batch}.parquet')

cols = ['follower_id', 'strong', 'weak']

merged = df[['follower_id', 'influencer_id', 'strong', 'weak']].merge(
    df1[['follower_id', 'influencer_id', 'strong', 'weak']], 
    on=['follower_id', 'influencer_id'], 
    suffixes=('_df', '_df1')
)

mismatches = merged[(merged['strong_df'] != merged['strong_df1']) | (merged['weak_df'] != merged['weak_df1'])]


print("follower_id sets match:", set(df['follower_id'].astype(str)) == set(df1['follower_id'].astype(str)))
print("strong match:", (merged['strong_df'] == merged['strong_df1']).all())
print("weak match:", (merged['weak_df'] == merged['weak_df1']).all())

follower_id sets match: True
strong match: True
weak match: True


## Batch 1: SA

In [37]:
country = 'SA'
path_save = f'../../data/randomization/{country}/influencers/'

batch = ''
tweets = pd.read_parquet(f"{path_save}tweets{batch}.parquet")

# Followers 
followers = pd.read_parquet(f"{path_save}followers{batch}.parquet")

# Likes
likes = pd.read_parquet(f"{path_save}likes{batch}.parquet")

# Replies
replies = pd.read_parquet(f"{path_save}replies{batch}.parquet")

# Participants TW
participants_tw = pd.read_parquet(f'{path_save}/confirmed_influencers{batch}.parquet')
participants_tw['author_id'] = participants_tw['id'].astype(str)
participants_tw = participants_tw.drop(['id'], axis = 1)
usernames_tw = list(participants_tw['handle'])
ids_tw = list(participants_tw['author_id'].astype(str))

get_followers_quality(
        participants_tw,
        followers,
        likes,
        replies,
        path_save, 
        batch = batch
    )

# Replicates: NO but the magnitude of the replication is probably due to a missing file that we scrapped live

df = pd.read_parquet(f'{path_save}followers_ties{batch}.parquet')
df1 = pd.read_parquet(f'{path_save}followers_ties_rep{batch}.parquet')

cols = ['follower_id', 'strong', 'weak']

merged = df[['follower_id', 'influencer_id', 'strong', 'weak']].merge(
    df1[['follower_id', 'influencer_id', 'strong', 'weak']], 
    on=['follower_id', 'influencer_id'], 
    suffixes=('_df', '_df1')
)

mismatches = merged[(merged['strong_df'] != merged['strong_df1']) | (merged['weak_df'] != merged['weak_df1'])]


print("follower_id sets match:", set(df['follower_id'].astype(str)) == set(df1['follower_id'].astype(str)))
print("strong match:", (merged['strong_df'] == merged['strong_df1']).all())
print("weak match:", (merged['weak_df'] == merged['weak_df1']).all())

mismatches = merged[(merged['strong_df'] != merged['strong_df1']) | (merged['weak_df'] != merged['weak_df1'])]
print(len(mismatches))

follower_id sets match: True
strong match: False
weak match: False
724


### Batch 2: SA

In [38]:
country = 'SA'
path_save = f'../../data/randomization/{country}/influencers/'

batch = '_batch2'
tweets = pd.read_parquet(f"{path_save}tweets{batch}.parquet")

# Followers 
followers = pd.read_parquet(f"{path_save}followers{batch}.parquet")

# Likes
likes = pd.read_parquet(f"{path_save}likes{batch}.parquet")

# Replies
replies = pd.read_parquet(f"{path_save}replies{batch}.parquet")

# Participants TW
participants_tw = pd.read_parquet(f'{path_save}/confirmed_influencers{batch}.parquet')
participants_tw['author_id'] = participants_tw['id'].astype(str)
participants_tw = participants_tw.drop(['id'], axis = 1)
usernames_tw = list(participants_tw['handle'])
ids_tw = list(participants_tw['author_id'].astype(str))

get_followers_quality(
        participants_tw,
        followers,
        likes,
        replies,
        path_save, 
        batch = batch
    )

# Replicates: YES

df = pd.read_parquet(f'{path_save}followers_ties{batch}.parquet')
df1 = pd.read_parquet(f'{path_save}followers_ties_rep{batch}.parquet')

cols = ['follower_id', 'strong', 'weak']

merged = df[['follower_id', 'influencer_id', 'strong', 'weak']].merge(
    df1[['follower_id', 'influencer_id', 'strong', 'weak']], 
    on=['follower_id', 'influencer_id'], 
    suffixes=('_df', '_df1')
)

mismatches = merged[(merged['strong_df'] != merged['strong_df1']) | (merged['weak_df'] != merged['weak_df1'])]


print("follower_id sets match:", set(df['follower_id'].astype(str)) == set(df1['follower_id'].astype(str)))
print("strong match:", (merged['strong_df'] == merged['strong_df1']).all())
print("weak match:", (merged['weak_df'] == merged['weak_df1']).all())

follower_id sets match: True
strong match: True
weak match: True


## Batch 2: KE

In [ ]:
country = 'KE'
path_save = f'../../data/randomization/{country}/influencers/'

batch = '_batch2'
tweets = pd.read_parquet(f"{path_save}tweets{batch}.parquet")

# Followers 
followers = pd.read_parquet(f"{path_save}followers{batch}.parquet")

# Likes
likes = pd.read_parquet(f"{path_save}likes{batch}.parquet")

# Replies
replies = pd.read_parquet(f"{path_save}replies{batch}.parquet")

# Participants TW
participants_tw = pd.read_parquet(f'{path_save}/confirmed_influencers{batch}.parquet')
participants_tw['author_id'] = participants_tw['id'].astype(str)
participants_tw = participants_tw.drop(['id'], axis = 1)
usernames_tw = list(participants_tw['handle'])
ids_tw = list(participants_tw['author_id'].astype(str))

get_followers_quality(
        participants_tw,
        followers,
        likes,
        replies,
        path_save, 
        batch = batch
    )

# Replicates: NOT YET MISSING INPUTS AND OUTPUTS

df = pd.read_parquet(f'{path_save}followers_ties{batch}.parquet')
df1 = pd.read_parquet(f'{path_save}followers_ties_rep{batch}.parquet')

cols = ['follower_id', 'strong', 'weak']

merged = df[['follower_id', 'influencer_id', 'strong', 'weak']].merge(
    df1[['follower_id', 'influencer_id', 'strong', 'weak']], 
    on=['follower_id', 'influencer_id'], 
    suffixes=('_df', '_df1')
)

mismatches = merged[(merged['strong_df'] != merged['strong_df1']) | (merged['weak_df'] != merged['weak_df1'])]


print("follower_id sets match:", set(df['follower_id'].astype(str)) == set(df1['follower_id'].astype(str)))
print("strong match:", (merged['strong_df'] == merged['strong_df1']).all())
print("weak match:", (merged['weak_df'] == merged['weak_df1']).all())